# Qui parle ? — démarrage

Retrouver la source (30 médias francophones) d'un texte d'actualité.
Ce notebook télécharge les données, entraîne un premier modèle (TF-IDF + régression
logistique, exactitude ≈ 0.42) et le dépose sur [le challenge](https://ml-arena.com/viewchallenge/178).

Il vous faut une clé API `mlk_user_…` (page **Profil** d'ml-arena.com).

In [ ]:
# scikit-learn 1.8 : la version du runtime de la plateforme (un modèle d'une autre version peut ne pas se recharger)
%pip install -q mlarena-sdk "scikit-learn==1.8.*" 

## 1. Les données

In [ ]:
import glob, os
from getpass import getpass
import mlarena

client = mlarena.connect(os.environ.get("MLARENA_API_KEY") or getpass("Clé API mlk_user_… : "))
if not glob.glob("data/qui_parle_textes_*.csv.gz"):
    client.download_dataset(178, "data")
DATA = sorted(glob.glob("data/qui_parle_textes_*.csv.gz"))[-1]   # l'instantané le plus récent
DATA

In [ ]:
import pandas as pd

df = pd.read_csv(DATA)
print(len(df), "textes,", df["day"].nunique(), "journées")
df[["day", "source_key", "text"]].sample(5, random_state=0)

Chaque matin, la plateforme envoie les textes de la journée écoulée, **au plus 20 par source**.
Le fichier, lui, contient tout : les volumes sont très inégaux, d'où `class_weight="balanced"` plus bas.

In [ ]:
df["source_key"].value_counts().plot.barh(figsize=(6, 7));

## 2. Valider comme la plateforme

On garde les **7 dernières journées** pour la validation (jamais un découpage au hasard :
les textes d'un même jour parlent des mêmes événements), et on tire chaque journée comme
l'évaluation : 20 textes au plus par source.

In [ ]:
days = sorted(df["day"].unique())
train, valid = df[df["day"] < days[-7]], df[df["day"] >= days[-7]]

def daily_samples(frame):
    for day, g in frame.groupby("day"):
        yield day, g.sample(frac=1, random_state=0).groupby("source_key").head(20)

print(len(train), "textes d'entraînement,", len(valid), "de validation")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def build_model():
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=300_000),
        LogisticRegression(C=10, max_iter=2000, class_weight="balanced"),
    )

model = build_model().fit(train["text"], train["source_key"])   # quelques minutes

In [ ]:
import numpy as np

scores = []
for day, s in daily_samples(valid):
    acc = (model.predict(s["text"]) == s["source_key"]).mean()
    scores.append(acc)
    print(day, f"{len(s)} textes  exactitude {acc:.3f}")
print(f"moyenne {np.mean(scores):.3f}  (hasard : 0.033)")

## 3. L'agent

La plateforme construit `Agent()` puis appelle `predict(request)` une fois par jour :
`request["texts"]` (600 textes au plus) et `request["classes"]` (les 30 clés). Il faut renvoyer
une liste de **même longueur** que `texts`. 60 s pour charger, 60 s pour prédire, 3 CPU, pas de réseau.

On ré-entraîne sur tout le fichier, puis on écrit `dist/` : `agent.py` + `model.joblib`.

In [ ]:
import joblib

model = build_model().fit(df["text"], df["source_key"])
os.makedirs("dist", exist_ok=True)
joblib.dump(model, "dist/model.joblib", compress=3);

In [ ]:
%%writefile dist/agent.py
import os
import joblib

HERE = os.path.dirname(os.path.abspath(__file__))


class Agent:
    def __init__(self):
        self.model = joblib.load(os.path.join(HERE, "model.joblib"))

    def predict(self, request):
        return [str(label) for label in self.model.predict(request["texts"])]

In [ ]:
# Vérification locale, comme le fait la plateforme
import importlib.util

spec = importlib.util.spec_from_file_location("agent", "dist/agent.py")
agent_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(agent_module)
agent = agent_module.Agent()
texts = valid["text"].tolist()[:600]
answer = agent.predict({"texts": texts, "classes": sorted(df["source_key"].unique())})
assert len(answer) == len(texts)
answer[:5]

## 4. Déposer

Chaque dépôt crée une nouvelle soumission. Elle est notée à partir du **deuxième** run de
05:00 UTC qui suit le dépôt (les textes du jour même ont pu être vus).

In [ ]:
client.submit(
    178,
    files=["dist/agent.py", "dist/model.joblib"],
    submission_name="tfidf-logreg",
    runtime={"framework": "scikit_learn_xgboost_lightgbm"},
    wait=True,
)

## Et ensuite ?

- **N-grammes de caractères** (`TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))`) + SVM linéaire : ≈ 0.61.
- **Un petit modèle de langue français** (DistilCamemBERT) affiné, runtime PyTorch : ≈ 0.75.
- **Le plafond de 20 textes par source** : répartir les textes entre sources plutôt que prendre le max (algorithme hongrois).